# 5 · Solving — Dirichlet dofs & static condensation

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=05-solving.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/05-solving.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>
:::

:::{dropdown} 🎭 Story — earning the Beast's trust
:class: storytelling

*You can cast a weak form, conjure geometry and wield functions on the mesh. The last
craft before the Beast truly trusts you is to understand the **solve** itself — how its
boundary is pinned, and how the system is tamed down to size.*
:::

**Solving** is the last craft, built up in three steps. First the *simplest* possible
linear system — an **L²-projection**, where **every** dof is an unknown — and we look at
the **sparse matrix** itself. Then we add **Dirichlet boundary conditions**: some dofs
become **prescribed**, splitting the matrix into **blocks**, and NGSolve solves it by
**lifting**. Finally, as a supplement, **static condensation** shrinks the system by
eliminating the element-internal dofs first. We *draw* every matrix as we go.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

mesh = Mesh(unit_square.GenerateMesh(maxh=0.4))

def to_dense(ngmat, n):
    """An NGSolve sparse matrix as a dense n×n NumPy array (these are small, for drawing)."""
    rows, cols, vals = ngmat.COO()
    A = np.zeros((n, n))
    A[np.asarray(rows, int), np.asarray(cols, int)] = np.asarray(vals, float)
    return A

def spy_blocks(ax, A, cut, labels, colors, title):
    """`spy` A and shade the 2×2 block structure induced by a dof split at index `cut`."""
    n = A.shape[0]
    ax.spy(A, markersize=2.2, color="#334155")
    seg = [0, cut, n]
    for i in range(2):
        for j in range(2):
            x0, y0 = seg[j], seg[i]
            w, h = seg[j + 1] - seg[j], seg[i + 1] - seg[i]
            ax.add_patch(patches.Rectangle((x0 - .5, y0 - .5), w, h, facecolor=colors[i][j],
                                           alpha=.22, edgecolor=colors[i][j], lw=1.2))
            ax.text(x0 + w / 2 - .5, y0 + h / 2 - .5, labels[i][j], ha="center", va="center",
                    color=colors[i][j], fontsize=11, weight="bold")
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

## 1. The simplest solve — an L²-projection

Forget boundaries for a moment. To **project** a function $g$ onto a finite element space
we solve $M\,\mathbf u=\mathbf b$ with the **mass matrix** $M_{ij}=\int\varphi_i\varphi_j$
and $b_i=\int g\,\varphi_i$. **Every** dof is an unknown — nothing is prescribed — so this
is *assemble → solve* in its purest form.

In [ ]:
fesP = H1(mesh, order=3)
u, v = fesP.TnT()
M = BilinearForm(u * v * dx).Assemble()
g = sin(5 * x) * y
gfp = GridFunction(fesP)
gfp.vec.data = M.mat.Inverse(fesP.FreeDofs(), inverse="sparsecholesky") \
               * LinearForm(g * v * dx).Assemble().vec
print(f"projected {fesP.ndof} dofs;   L2 error ‖u−g‖ = {sqrt(Integrate((gfp - g)**2, mesh)):.1e}")

The matrix is **sparse** and **symmetric**: each basis function overlaps only its
neighbours, so $M$ has just a handful of entries per row. Let us actually *look* at it.

In [ ]:
Mdense = to_dense(M.mat, fesP.ndof)
fig, ax = plt.subplots(figsize=(4.3, 4.3))
ax.spy(Mdense, markersize=2.2, color="#334155")
ax.set_title(f"mass matrix M  ({fesP.ndof}×{fesP.ndof}, {int((Mdense != 0).sum())} nonzeros)\n"
             f"every dof free — solve M u = b", fontsize=10)
ax.set_xticks([]); ax.set_yticks([]); plt.tight_layout(); plt.show()

## 2. Free dofs & Dirichlet dofs — a block system

Now solve **Poisson** $-\Delta u=f$ with $u=g$ on the boundary. The `dirichlet` flag marks
the boundary dofs **essential** — *prescribed*, not solved for. `fes.FreeDofs()` is a
bit-array, `True` on the **free** dofs. If we **reorder** the dofs *free first, Dirichlet
last*, the stiffness matrix shows a clean **2×2 block structure**
$A=\bigl(\begin{smallmatrix}A_{ff}&A_{fd}\\A_{df}&A_{dd}\end{smallmatrix}\bigr)$:

In [ ]:
fes = H1(mesh, order=3, dirichlet="bottom|right|top|left")
u, v = fes.TnT()
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
f = LinearForm(1 * v * dx).Assemble()

free = fes.FreeDofs()
fmask = np.array([bool(free[i]) for i in range(fes.ndof)])
nf = int(fmask.sum())
perm = np.concatenate([np.where(fmask)[0], np.where(~fmask)[0]])     # free dofs first
print(f"{fes.ndof} dofs  =  {nf} free  +  {fes.ndof - nf} Dirichlet")

fig, ax = plt.subplots(figsize=(4.7, 4.7))
A = to_dense(a.mat, fes.ndof)
spy_blocks(ax, A[perm][:, perm], nf,
           [["A_ff", "A_fd"], ["A_df", "A_dd"]],
           [["#10b981", "#f59e0b"], ["#f59e0b", "#ef4444"]],
           f"stiffness, reordered: {nf} free + {fes.ndof - nf} Dirichlet")
plt.tight_layout(); plt.show()

**The NGSolve-style solve — lifting.** The prescribed values $\mathbf u_d$ feed into the
free equations through $A_{fd}$, so they cannot be ignored. We **set** the boundary values,
carry their effect to the right-hand side as a **residual** $\mathbf r=\mathbf b-A\,\mathbf u$,
and solve only the free block, $A_{ff}\,\Delta\mathbf u_f=\mathbf r_f$:

In [ ]:
g = sin(4 * x) * y
gfu = GridFunction(fes)
gfu.Set(g, BND)                                      # u_d : prescribe u = g on the boundary
r = f.vec - a.mat * gfu.vec                          # residual = b − A u  (carries A_fd u_d)
gfu.vec.data += a.mat.Inverse(free, inverse="sparsecholesky") * r   # solve on the free dofs
Draw(gfu, mesh, "u with u = g on the boundary")

## Supplementary — static condensation

*(Skip if short on time.)* A high-order space has **internal** dofs (bubbles inside one
element) that couple **only** to that element. Reorder the free dofs *local (internal)
first, coupling last*: the **local block $A_{ll}$ is block-diagonal** — no two elements
share an internal dof — so it inverts **element-by-element**. Eliminating it leaves a
smaller **Schur** system on the coupling dofs. That is *static condensation*.

In [ ]:
ct = [str(fes.CouplingType(i)).split(".")[-1] for i in range(fes.ndof)]
local = [i for i in np.where(fmask)[0] if ct[i] == "LOCAL_DOF"]
coupl = [i for i in np.where(fmask)[0] if ct[i] != "LOCAL_DOF"]
permc = np.array(local + coupl)
fig, ax = plt.subplots(figsize=(4.7, 4.7))
spy_blocks(ax, A[permc][:, permc], len(local),
           [["A_ll", "A_lc"], ["A_cl", "A_cc"]],
           [["#3b82f6", "#f59e0b"], ["#f59e0b", "#10b981"]],
           f"free dofs: {len(local)} local + {len(coupl)} coupling   (A_ll block-diagonal)")
plt.tight_layout(); plt.show()

NGSolve does the bookkeeping with **`condense=True`**: it factors out the local block, so
the global solve runs on the much smaller `FreeDofs(coupling=True)` set and then
reconstructs the internal dofs locally — same answer, smaller system.

In [ ]:
ac = BilinearForm(grad(u) * grad(v) * dx, condense=True).Assemble()
n_couple = sum(1 for d in fes.FreeDofs(coupling=True) if d)
print(f"global solve shrinks from {nf} free dofs to {n_couple} coupling dofs")

inv = ((IdentityMatrix() + ac.harmonic_extension)
       @ ac.mat.Inverse(fes.FreeDofs(coupling=True), inverse="sparsecholesky")
       @ (IdentityMatrix() + ac.harmonic_extension_trans) + ac.inner_solve)
gfc = GridFunction(fes); gfc.Set(g, BND)
gfc.vec.data += ac.harmonic_extension * gfc.vec
gfc.vec.data += inv * (f.vec - ac.mat * gfc.vec)
print(f"condensed vs plain solve differ by {sqrt(Integrate((gfu - gfc)**2, mesh)):.1e} (same answer)")

:::{dropdown} 📚 Further reading
:class: further-reading

- **Dirichlet boundary conditions** — i-tutorial
  [1.3 Dirichlet](https://docu.ngsolve.org/latest/i-tutorials/unit-1.3-dirichlet/dirichlet.html).
- **Static condensation** — i-tutorial
  [1.4 Static condensation](https://docu.ngsolve.org/latest/i-tutorials/unit-1.4-staticcond/staticcond.html).
:::

:::{dropdown} 🧠 Quiz — when is static condensation worth it?
:class: quiz
Most when the space has **many internal dofs per element** — high polynomial order, or
**bubble/DG-like** spaces, especially in 3D where internal dofs dominate. The coupling
system can be several times smaller, so a (sparse) direct factorisation is much cheaper,
and iterative solvers see a better-conditioned operator. For lowest order ($p=1$, no
interior dofs) there is nothing to condense. The same `condense` machinery underlies
**hybrid** methods (notebook 6's solver toolbox goes further).
:::

**Next:** the solve above used a direct factorisation. Unit 6 — the close of Part I —
opens the **solver toolbox**: the iterative methods and preconditioners that scale to
problems a direct solver can no longer swallow.

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "06-linear-solvers", "6 · The solver toolbox 🛠"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))